# Lab 5 Working with Data on the Command Line

Lab 5's matrix class is the main assignment. This shorter piece is what we work through in
the lab session, and it is a different skill: finding out what is in a data file, and
getting it into shape, before any Python is involved.

Lecture 11 starts from a real grades file that is a mess — a line of notes where the header
should be, words like `no solution` in columns that should hold numbers, scores written as
`68-sent` — and cleans it with pandas. The same jobs, and almost always the *first* look at
a new file, can be done faster with the Unix tools already on your machine. They also work
when the file is far too big to open.

Lab 3 covered moving around the filesystem: `ls`, `cd`, `mkdir`, `cp`, `mv`. Here we chain
commands together to ask questions of a file and to write a cleaned copy of it.

## The data

Nothing to download. Everything is in the repository you pulled:

| File | What it is |
|---|---|
| `Labs/Lab.5/bank-churn.csv` | 20,000 bank customers: name, country, age, balance, and whether they left. Mixed text and numbers. |
| `Labs/Lab.5/heart.csv` | 303 patients, 14 numeric columns, and a 0/1 diagnosis. |
| `Labs/Lab.5/2012_Workplace_Fatalities_by_State.csv` | Workplace fatalities per US state. Awkward on purpose — see the last exercise. |
| `Lectures/Lecture.11/Data-1401-Grades.csv` | The messy grades file from Lecture 11. |

The bank file is a 20,000-row extract of a Kaggle playground dataset (Binary Classification
with a Bank Churn Dataset, 2024); the heart and fatalities files are included with Lab 5.

## Finding the commands

`man <command>` is the manual for any of these: space scrolls, `q` quits. Some resources
found by searching:

* Paths and wildcards: https://www.warp.dev/terminus/linux-wildcards
* Introduction to the shell: https://github-pages.ucl.ac.uk/RCPSTrainingMaterials/HPCandHTCusingLegion/2_intro_to_shell.html
* Chaining commands: https://www.geeksforgeeks.org/chaining-commands-in-linux/
* Piping: https://www.geeksforgeeks.org/piping-in-unix-or-linux/
* Using `sed`: https://www.geeksforgeeks.org/sed-command-linux-set-2/
* Cheat sheet: https://cheatography.com/davechild/cheat-sheets/linux-command-line/

## How to work

Use a real command prompt: the WSL Ubuntu terminal on Windows, Terminal on macOS. When a
command does what you want, paste it into the cell under the exercise. A cell starting with
`!` runs a shell command from inside Jupyter — that is how Lecture 11 ran `cat` and `wc` —
so your pasted commands will run here too.

Three things to know first:

* `>` sends output into a file, replacing what was there.
* `>>` appends to the end of a file instead.
* `|` (a pipe) feeds one command's output straight into the next.


## Exercise 0: Somewhere to work

Make a directory for this lab inside your fork and copy the four CSV files into it. Work on
the copies, so a mistake never touches the originals and `git status` stays clean.

Hints: `mkdir`, then `cp`. A wildcard saves typing, and `cp` takes several files at once if
the last argument is a directory.

In [4]:
!mkdir -p lab_csv
!cp *.csv lab_csv/
!ls lab_csv


2012_Workplace_Fatalities_by_State.csv heart.csv
bank-churn.csv


## Exercise 1: First look at a file you have never seen

You have `bank-churn.csv` and know nothing about it. Answer these at the prompt:

1. How big is the file, and how many rows does it have?
2. What are the column names, and how many columns are there?
3. What do the first few rows actually look like?

Hints: `du -h` for size, `wc -l` for lines. `head -1` gets the header; piping it into
`tr ',' '\n'` puts each column name on its own line, and piping *that* into `nl` numbers
them — which you will want in the later exercises, where `cut` asks for column numbers.

In [5]:
!du -h lab_csv/bank-churn.csv
!wc -l lab_csv/bank-churn.csv

!head -1 lab_csv/bank-churn.csv | tr ',' '\n' | nl
!head -5 lab_csv/bank-churn.csv

1.4M	lab_csv/bank-churn.csv
   20001 lab_csv/bank-churn.csv
     1	id
     2	CustomerId
     3	Surname
     4	CreditScore
     5	Geography
     6	Gender
     7	Age
     8	Tenure
     9	Balance
    10	NumOfProducts
    11	HasCrCard
    12	IsActiveMember
    13	EstimatedSalary
    14	Exited
id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.0,2,1.0,0.0,181449.97,0
1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.0,2,1.0,1.0,49503.5,0
2,15694510,Hsueh,678,France,Male,40.0,10,0.0,2,1.0,0.0,184866.69,0
3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0


## Exercise 2: Split a big file into three

Files too big to open are routine, and a common first move is to cut one into pieces.

Make 3 new CSV files, each with about a third of `bank-churn.csv`'s rows, and **each
starting with the header line**, so every piece is a readable CSV on its own.

Hints:

* `head -1 ... > part1.csv` starts a file with just the header. Do that three times.
* `wc -l` tells you how many rows there are to divide up. Remember one of them is the header.
* `head -N file | tail -M` selects a range: M lines ending at line N.
* Append each range to the right file with `>>`.
* Check your work: the three files' rows should add back up, `wc -l` on all three at once.

In [11]:
import subprocess

file = "lab_csv/bank-churn.csv"

total = int(subprocess.check_output(["wc", "-l", file]).decode().split()[0]) - 1

third = total // 3

print("Total data rows:", total)
print("Rows per part (approximately):", third)

!head -1 lab_csv/bank-churn.csv > lab_csv/part1.csv
!head -1 lab_csv/bank-churn.csv > lab_csv/part2.csv
!head -1 lab_csv/bank-churn.csv > lab_csv/part3.csv

!tail -n +2 lab_csv/bank-churn.csv | head -n {third} >> lab_csv/part1.csv

!tail -n +2 lab_csv/bank-churn.csv | head -n {third * 2} | tail -n {third} >> lab_csv/part2.csv

!tail -n +2 lab_csv/bank-churn.csv | tail -n +{third * 2 + 1} >> lab_csv/part3.csv

print("\nRows in each file:")
!wc -l lab_csv/part1.csv lab_csv/part2.csv lab_csv/part3.csv

print("\nHeaders:")
!head -1 lab_csv/part1.csv
!head -1 lab_csv/part2.csv
!head -1 lab_csv/part3.csv


Total data rows: 20000
Rows per part (approximately): 6666
tail: stdout: Broken pipe
tail: stdout: Broken pipe

Rows in each file:
    6667 lab_csv/part1.csv
    6667 lab_csv/part2.csv
    6669 lab_csv/part3.csv
   20003 total

Headers:
id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited


## Exercise 3: Select rows by what they contain, and count them

`grep <pattern> <file>` prints the lines that match; `grep -v` prints the ones that don't;
`grep -c` counts instead of printing.

1. Write a CSV holding only the German customers, header line included.
2. What fraction of all customers are German?
3. Of the German customers, what fraction left the bank? The last column, `Exited`, is 1 for
   customers who left. A row for a German customer who left ends in `,1`.

Hints: pipe `grep` into `grep` to require two things at once, and into `wc -l` to count.
Watch out for `Germany` appearing in a surname — check whether that can happen here, and say
how you checked.

In [15]:
import subprocess

file = "lab_csv/bank-churn.csv"

header = subprocess.check_output(["head", "-1", file], text=True).strip().split(",")

geo_col = header.index("Geography") + 1
surname_col = header.index("Surname") + 1

print("Geography column:", geo_col)
print("Surname column:", surname_col)

!head -1 lab_csv/bank-churn.csv > lab_csv/german_customers.csv

!tail -n +2 lab_csv/bank-churn.csv | awk -F, -v c={geo_col} '$c=="Germany"' >> lab_csv/german_customers.csv

print("\nFirst five rows of German customers:")
!head -5 lab_csv/german_customers.csv

total = int(subprocess.check_output(f"tail -n +2 {file} | wc -l", shell=True, text=True).strip())

german = int(subprocess.check_output("tail -n +2 lab_csv/german_customers.csv | wc -l",shell=True, text=True).strip())

print("\nQuestion 2:")
print("Total customers:", total)
print("German customers:", german)
print("Fraction German:", german / total)
print("Percentage German:", round(german / total * 100, 2), "%")

left = int(subprocess.check_output("tail -n +2 lab_csv/german_customers.csv | grep -c ',1$'",shell=True, text=True).strip())

print("\nQuestion 3:")
print("German customers who left:", left)
print("Fraction who left:", left / german)
print("Percentage who left:", round(left / german * 100, 2), "%")

surname_matches = int(subprocess.check_output(f"tail -n +2 {file} | cut -d',' -f{surname_col} | grep -c 'Germany' || true",shell=True, text=True).strip())

print("\nSurname check:")
print("Surnames containing 'Germany':", surname_matches)

if surname_matches == 0:
    print("No surnames contain 'Germany'.")
else:
    print("Some surnames contain 'Germany'.")




Geography column: 5
Surname column: 3

First five rows of German customers:
id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
5,15771669,Genovese,588,Germany,Male,36.0,4,131778.58,1,1.0,0.0,136024.31,1
9,15591721,Cattaneo,583,Germany,Male,40.0,4,81274.33,1,1.0,1.0,170843.07,0
10,15635097,Fokine,599,Germany,Female,27.0,6,161801.47,2,1.0,0.0,109184.24,0
11,15674671,Chiu,687,Germany,Male,40.0,3,90432.92,1,1.0,0.0,1676.92,0

Question 2:
Total customers: 20000
German customers: 4155
Fraction German: 0.20775
Percentage German: 20.77 %

Question 3:
German customers who left: 1561
Fraction who left: 0.3756919374247894
Percentage who left: 37.57 %

Surname check:
Surnames containing 'Germany': 0
No surnames contain 'Germany'.


## Exercise 4: Counting lines is not counting matches

In the grades file, some exam scores are written `68-sent`, the grader's note that the score
had been emailed to the student. Count them.

Careful: `wc -l` counts *lines*, and a line can hold more than one `-sent`. Ask `grep` for
the number of matches rather than the number of matching lines — read `man grep` for `-o`
and for `-c`, and work out which one you need here. Compare the two answers on this file. They agree, which tells you something about the file: no line carries two of them. Describe a row that would make them disagree.

In [22]:
import subprocess

file = "../../Lectures/Lecture.11/Data-1401-Grades.csv"

lines = subprocess.run(["grep", "-c", "--", "-sent", file],capture_output=True, text=True)
line_count = int(lines.stdout.strip() or 0)

matches = subprocess.run(["grep", "-o", "--", "-sent", file],capture_output=True, text=True)
match_count = len(matches.stdout.splitlines())

print("Lines containing -sent:", line_count)
print("Total occurrences of -sent:", match_count)
print("Do the counts agree?", line_count == match_count)

print("\nExplanation:")
print("grep -c counts matching lines.")
print("grep -o counts individual matches.")
print("The counts agree if no line contains -sent more than once.")
print("A row with 68-sent and 75-sent would make the counts disagree.")


Lines containing -sent: 12
Total occurrences of -sent: 12
Do the counts agree? True

Explanation:
grep -c counts matching lines.
grep -o counts individual matches.
The counts agree if no line contains -sent more than once.
A row with 68-sent and 75-sent would make the counts disagree.


## Exercise 5: Clean a file with `sed`

`sed` replaces text as it passes through. Work on the grades file:

1. Its first line is a row of notes about how each lab was graded, not the header. Write a
   new file with everything **except** that first line.
2. In that new file, make these replacements, and write the result to another CSV:
   * `no solution` becomes empty
   * `did not submit the Lab` becomes empty
   * `did not submit the exam` becomes empty
   * `68-sent` becomes `68`, and likewise for every other score written that way

Hints:

* For part 1, `tail` counts from the start of the file if you ask it the right way:
  `man tail`, look for `-n +K`.
* The form is `sed 's/old/new/g' file`; `g` means every match on the line, not just the first.
* Chain several `sed` commands with pipes.
* The last replacement isn't a fixed string, since the number changes from row to row. You
  want to remove the `-sent` and keep the number, so match only `-sent`.

Compare your part 1 output against `Lectures/Lecture.11/Data-1401-Grades-Fixed.csv`, the
file Lecture 11 hands you: `cmp` or `diff` will tell you whether they're identical. You have
just made it yourself.

This is the same cleaning Lecture 11 does in pandas. Neither way is the right one. The shell
is quicker for a look and a one-off fix; pandas is where you go when the cleaning becomes
part of an analysis you will run again.

In [24]:
from pathlib import Path
import subprocess

file = "../../Lectures/Lecture.11/Data-1401-Grades.csv"

folder = Path("lab_csv")
folder.mkdir(exist_ok=True)

part1 = folder / "grades_without_first_line.csv"
cleaned = folder / "grades_cleaned.csv"

with open(part1, "w") as output:
    subprocess.run(["tail", "-n", "+2", file],stdout=output,check=True)

print("Part 1 saved:", part1)

with open(cleaned, "w") as output:
    subprocess.run(
        ["sed","-e", "s/no solution//g","-e", "s/did not submit the Lab//g","-e", "s/did not submit the exam//g","-e", "s/-sent//g",str(part1)],stdout=output,check=True
    )

print("Part 2 saved:", cleaned)

print("\nFirst five rows:")
subprocess.run(["head", "-5", str(cleaned)], check=True)

print("\nCheck for remaining unwanted text:")
result = subprocess.run(
    ["grep", "-nE","no solution|did not submit the Lab|did not submit the exam|-sent",str(cleaned)],capture_output=True, text=True
)

print(result.stdout if result.stdout else "No unwanted text found.")

reference = "Lectures/Lecture.11/Data-1401-Grades-Fixed.csv"

if Path(reference).exists():
    result = subprocess.run(
        ["cmp", "-s", str(part1), reference]
    )
    print("\nPart 1 matches reference:", result.returncode == 0)
else:
    print("\nReference file not found.")


Part 1 saved: lab_csv/grades_without_first_line.csv
Part 2 saved: lab_csv/grades_cleaned.csv

First five rows:
Lab 1,Lab 2,Lab 3,Lab 4,Exam 1,Lab 5,,Lab 7,Exam 2
,0,0,,0,0,0,0,0
,,-,,,,,,
100,96,46,,68,,,,
Completed the lab,"ex4- attempted,wrong solution -2

Check for remaining unwanted text:
No unwanted text found.

Reference file not found.


## Exercise 6: Drop the columns that shouldn't be there

`bank-churn.csv` starts with `id`, `CustomerId` and `Surname`. None of them tells you
anything about whether a customer leaves — an id is just bookkeeping, and if a model appears
to learn something from one, it has learned about how the file was built, not about
customers. Dropping them is a real step in preparing data.

Write a new CSV without those three columns.

Hints: `cut -f` selects fields by number and `-d ','` sets the separator, which a CSV needs.
`cut` takes ranges: `-f 4-` means "from the fourth field on".

In [25]:
from pathlib import Path
import subprocess

file = Path("lab_csv/bank-churn.csv")

if not file.exists():
    file = Path("bank-churn.csv")

if not file.exists():
    raise FileNotFoundError("bank-churn.csv not found. Check your folder.")

folder = Path("lab_csv")
folder.mkdir(exist_ok=True)

output = folder / "bank-churn-cleaned.csv"

with open(output, "w") as f:
    subprocess.run(
        ["cut", "-d", ",", "-f", "4-", str(file)],
        stdout=f,
        check=True
    )

print("New CSV created:", output)

print("\nOriginal columns:")
subprocess.run(["head", "-1", str(file)], check=True)

print("\nColumns after removing first three:")
subprocess.run(["head", "-1", str(output)], check=True)

print("\nRow counts:")
subprocess.run(["wc", "-l", str(file), str(output)], check=True)

print("\nFirst five rows of cleaned file:")
subprocess.run(["head", "-5", str(output)], check=True)


New CSV created: lab_csv/bank-churn-cleaned.csv

Original columns:
id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited

Columns after removing first three:
CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited

Row counts:
   20001 lab_csv/bank-churn.csv
   20001 lab_csv/bank-churn-cleaned.csv
   40002 total

First five rows of cleaned file:
CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
668,France,Male,33.0,3,0.0,2,1.0,0.0,181449.97,0
627,France,Male,33.0,1,0.0,2,1.0,1.0,49503.5,0
678,France,Male,40.0,10,0.0,2,1.0,0.0,184866.69,0
581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0


CompletedProcess(args=['head', '-5', 'lab_csv/bank-churn-cleaned.csv'], returncode=0)

## Exercise 7: Arithmetic on a column

Use `heart.csv`, which is all numbers.

1. Compute each patient's `trestbps + chol + thalach` (resting blood pressure, cholesterol,
   maximum heart rate) and write the totals to a file.
2. How many patients have `target` equal to 1?

Hints:

* `cut` gets you just the columns you want. Use the numbered header from Exercise 1.
* `tr ',' '+'` turns `145,233,150` into `145+233+150`.
* `bc` is a calculator that reads expressions and prints answers. Pipe into it.
* The header line is text, not numbers, so drop it first, the way you did in Exercise 5.
* `heart.csv` begins with an invisible marker (a byte-order mark) before `age`. It does no
  harm here, but if a command behaves oddly on the first column, that is why.

In [26]:
from pathlib import Path
import subprocess
import shutil

file = Path("lab_csv/heart.csv")

if not file.exists():
    file = Path("heart.csv")

if not file.exists():
    raise FileNotFoundError("heart.csv not found. Check your folder.")

folder = Path("lab_csv")
folder.mkdir(exist_ok=True)

output = folder / "heart_totals.txt"

with open(file, encoding="utf-8-sig") as f:
    header = f.readline().strip().split(",")

print("Column names:", header)

bp_col = header.index("trestbps") + 1
chol_col = header.index("chol") + 1
hr_col = header.index("thalach") + 1
target_col = header.index("target") + 1

print("\nColumn numbers:")
print("trestbps:", bp_col)
print("chol:", chol_col)
print("thalach:", hr_col)
print("target:", target_col)

cut_result = subprocess.run(
    ["cut", "-d", ",", "-f",
     f"{bp_col},{chol_col},{hr_col}", str(file)],
    capture_output=True, text=True, check=True
)

expressions = cut_result.stdout.splitlines()[1:]
expressions = [row.replace(",", "+") for row in expressions]

if shutil.which("bc"):
    result = subprocess.run(
        ["bc"],
        input="\n".join(expressions) + "\n",
        capture_output=True, text=True, check=True
    )
    totals = result.stdout.splitlines()
else:
    print("\nbc is not installed. Using Python for calculations.")
    totals = [
        str(sum(int(x) for x in row.split("+")))
        for row in expressions
    ]

with open(output, "w") as f:
    f.write("\n".join(totals) + "\n")

print("\nQuestion 1:")
print("Totals saved to:", output)
print("Number of patient totals:", len(totals))
print("First five totals:", totals[:5])

target_result = subprocess.run(
    ["cut", "-d", ",", "-f", str(target_col), str(file)],
    capture_output=True, text=True, check=True
)

targets = target_result.stdout.splitlines()[1:]
count = sum(value.strip() == "1" for value in targets)

print("\nQuestion 2:")
print("Number of patients with target = 1:", count)


Column names: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

Column numbers:
trestbps: 4
chol: 5
thalach: 8
target: 14

Question 1:
Totals saved to: lab_csv/heart_totals.txt
Number of patient totals: 303
First five totals: ['528', '567', '506', '534', '637']

Question 2:
Number of patients with target = 1: 165


## Exercise 8: Sort

1. Sort `bank-churn.csv` by `Balance`, largest first, keeping the header at the top, and
   write the result to a new CSV.
2. Who are the 5 customers with the largest balances, and what countries are they in?

Hints: `sort` with `-n` (numeric), `-r` (reverse), `-t ','` (the separator) and `-k` (which
field). `man sort` is worth reading for `-k`, the fiddly one. Keep the header out of the
sort the same way as before, then put it back with `>` and `>>`.

In [27]:
from pathlib import Path
import subprocess

file = Path("lab_csv/bank-churn.csv")

if not file.exists():
    file = Path("bank-churn.csv")

if not file.exists():
    raise FileNotFoundError("bank-churn.csv not found.")

header = file.open().readline().strip().split(",")

balance_col = header.index("Balance") + 1
surname_col = header.index("Surname") + 1
geography_col = header.index("Geography") + 1

print("Balance column:", balance_col)
print("Surname column:", surname_col)
print("Geography column:", geography_col)

folder = Path("lab_csv")
folder.mkdir(exist_ok=True)

output = folder / "bank-churn-sorted.csv"

with open(output, "w") as f:
    subprocess.run(
        ["head", "-1", str(file)],
        stdout=f, check=True
    )

with open(output, "a") as f:
    data = subprocess.run(
        ["tail", "-n", "+2", str(file)],
        capture_output=True, check=True
    )

    subprocess.run(
        ["sort", "-t", ",", "-k",
         f"{balance_col},{balance_col}", "-nr"],
        input=data.stdout,
        stdout=f,
        check=True
    )

print("\nSorted CSV saved to:", output)

print("\nTop 5 customers by Balance:")
print("Surname | Country | Balance")

with open(output) as f:
    next(f)
    for i in range(5):
        row = next(f).strip().split(",")
        print(
            row[surname_col - 1], "|",
            row[geography_col - 1], "|",
            row[balance_col - 1]
        )

print("\nFirst five rows of sorted CSV:")
subprocess.run(["head", "-5", str(output)], check=True)


Balance column: 9
Surname column: 3
Geography column: 5

Sorted CSV saved to: lab_csv/bank-churn-sorted.csv

Top 5 customers by Balance:
Surname | Country | Balance
Ibezimako | Spain | 250898.09
Nebeuwa | Spain | 238387.56
Nott | France | 222267.63
Thomson | Germany | 212696.32
Udinesi | France | 212314.03

First five rows of sorted CSV:
id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
12268,15757408,Ibezimako,655,Spain,Male,38.0,3,250898.09,3,1.0,1.0,81054.0,0
8949,15587596,Nebeuwa,572,Spain,Male,31.0,5,238387.56,3,1.0,0.0,21646.2,1
3968,15768451,Nott,681,France,Female,40.0,8,222267.63,1,1.0,0.0,186281.0,1
16904,15620268,Thomson,495,Germany,Male,43.0,3,212696.32,1,1.0,0.0,115268.86,0


CompletedProcess(args=['head', '-5', 'lab_csv/bank-churn-sorted.csv'], returncode=0)

## Exercise 9: Where the command line gives up

Open `2012_Workplace_Fatalities_by_State.csv` in a text editor and look at the first few
lines. Then, at the prompt:

1. How many lines does `wc -l` report?
2. How many US states are in the file?
3. Print the first column with `cut -d ',' -f 1`. What do you get?

The two counts in 1 and 2 disagree, and the `cut` output is wrong. Work out why, and write
a sentence or two explaining it. Two things in this file break the assumption every tool you
used today makes, that a line is a row and a comma separates fields.

This is the real lesson of the lab. These tools are fast and they are always there, but they
treat a CSV as lines of text. When the file's own format fights back, stop, and use something
that understands CSV — which is where `pandas.read_csv` comes in, in Lecture 11 and Lab 7.

In [28]:
from pathlib import Path
import subprocess
import csv

file = Path("lab_csv/2012_Workplace_Fatalities_by_State.csv")

if not file.exists():
    file = Path("2012_Workplace_Fatalities_by_State.csv")

if not file.exists():
    raise FileNotFoundError(
        "2012_Workplace_Fatalities_by_State.csv not found."
    )

print("Question 1: Number of lines reported by wc -l")

subprocess.run(
    ["wc", "-l", str(file)],
    check=True
)

print("\nQuestion 2: Number of states/records")

with open(file, newline="", encoding="utf-8-sig") as f:
    rows = list(csv.reader(f))

print("Total CSV records (including header):", len(rows))
print("Data records (excluding header):", len(rows) - 1)

print("\nQuestion 3: First column using cut")

result = subprocess.run(
    ["cut", "-d", ",", "-f", "1", str(file)],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout[:2000])

print("\nFirst column using Python's CSV reader:")

for row in rows[:10]:
    if row:
        print(row[0])

print("\nExplanation:")
print("wc -l counts physical lines, not actual CSV records.")
print("Some CSV fields contain line breaks inside quotation marks.")
print("Also, some fields contain commas inside quotation marks.")
print("The cut command treats every comma as a separator, even quoted ones.")
print("Python's csv.reader handles quoted commas and line breaks correctly.")
print("This is why the command-line results can be incorrect.")

print("\nConclusion:")
print("For complicated CSV files, use a CSV-aware reader such as pandas.read_csv.")


Question 1: Number of lines reported by wc -l
     105 lab_csv/2012_Workplace_Fatalities_by_State.csv

Question 2: Number of states/records
Total CSV records (including header): 55
Data records (excluding header): 54

Question 3: First column using cut
State
"South Carolina
(33.99882060100049
"West Virginia
(38.665511497000466
"Massachusetts
(42.27687306500047
"Tennessee
(35.680943063000484
"Oklahoma
(35.472034350000456
"Illinois
(40.48501278700047
"Nebraska
(41.64104043900045
"Delaware
(39.00883351400046
"Hawaii
(21.30485166200043
"Iowa
(42.469404401000475
"Arizona
(34.865973091000455
"Florida
(28.932042899000464
"Virginia
(37.54268075100049
"Missouri
(38.63579372300046
"Michigan
(44.66131575600048
"Indiana
(39.76691364600049
"North Carolina
(35.46622388600048
"New Hampshire
(43.6559537330005
"New Mexico
(34.52088247800049
"Pennsylvania
(40.79373106100047
"South Dakota
(44.35313342000046
"New York
(42.82700023900048
"Utah
(39.36070374600047
"Maine
(45.254228663000504
"Montana
(47.0665

## Submitting

Same as every lab: copy this notebook to a new name before working in it, and push the copy
to your fork.

```bash
cp Labs/Lab.5/Lab.5.CommandLine.ipynb Labs/Lab.5/Lab.5.CommandLine.solution.ipynb
```

Each exercise's cell should hold the command or commands you used, and where the exercise
asks a question — a fraction, a count, an explanation — write the answer next to them.

**Do not commit the CSV files you create.** They are working files; your commands are the
submission.
